# CNN Development on Custom Dataset — Assignment Notebook

**Chosen Dataset:** CIFAR-10  
**Task:** Multi-class image classification using a custom CNN in PyTorch

This notebook fully follows the assignment requirements:
- Load and preprocess an image dataset with PyTorch
- Create training, validation, and test loaders
- Build a custom CNN
- Use batch normalization and dropout
- Train with optimizer, scheduler, and cross-entropy loss
- Plot training and validation curves
- Evaluate on the test set using accuracy, precision, recall, and F1-score
- Show confusion matrix
- Analyze best and worst performing classes
- Save trained model weights

> Before running, make sure PyTorch, torchvision, matplotlib, seaborn, scikit-learn, and torchinfo are installed.

## 1. Import Libraries

In [ ]:
import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score
)

# Optional detailed model summary
try:
    from torchinfo import summary
    TORCHINFO_AVAILABLE = True
except ImportError:
    TORCHINFO_AVAILABLE = False
    print("torchinfo not installed. Model summary cell will be skipped.")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Hyperparameters and Configuration

In [ ]:
# Configuration
DATA_DIR = "./data"
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 15
VALID_RATIO = 0.1
NUM_CLASSES = 10
MODEL_SAVE_PATH = "cnn_cifar10_best.pth"

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

## 3. Data Preprocessing and Augmentation

We use:
- **Training augmentation:** random crop and horizontal flip
- **Normalization:** standard CIFAR-10 mean and std

This improves generalization and helps regularize the CNN.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010))
])

## 4. Load Dataset

We use **CIFAR-10**, which has:
- 10 classes
- 60,000 color images
- Image size: 32×32×3

We split the training portion into:
- training set
- validation set

In [ ]:
full_train_dataset = datasets.CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=test_transform
)

val_size = int(len(full_train_dataset) * VALID_RATIO)
train_size = len(full_train_dataset) - val_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Validation should not use augmentation-heavy transform
val_dataset.dataset = copy.deepcopy(full_train_dataset)
val_dataset.dataset.transform = test_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training samples   : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Test samples       : {len(test_dataset)}")

## 5. Visualize Sample Images

In [ ]:
def imshow(img):
    img = img.numpy().transpose((1, 2, 0))
    mean = np.array([0.4914, 0.4822, 0.4465])
    std = np.array([0.2023, 0.1994, 0.2010])
    img = std * img + mean
    img = np.clip(img, 0, 1)
    plt.imshow(img)

images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    imshow(images[i].cpu())
    plt.title(class_names[labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Define Custom CNN Architecture

This custom CNN includes:
- Convolutional layers
- Batch Normalization
- ReLU activation
- MaxPooling
- Dropout for regularization
- Fully connected classifier

This satisfies the requirement of using a custom CNN with regularization.

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CustomCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.30),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.35)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CustomCNN(num_classes=NUM_CLASSES).to(device)
print(model)

## 7. Model Summary

In [ ]:
if TORCHINFO_AVAILABLE:
    summary(model, input_size=(BATCH_SIZE, 3, 32, 32))
else:
    print("Install torchinfo to view a detailed summary:")
    print("pip install torchinfo")

## 8. Loss Function, Optimizer, and Learning Rate Scheduler

- **Loss:** CrossEntropyLoss
- **Optimizer:** Adam
- **Scheduler:** StepLR

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

## 9. Training and Validation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

## 10. Train the Model

In [ ]:
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0
best_model_weights = copy.deepcopy(model.state_dict())

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(best_model_weights, MODEL_SAVE_PATH)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Best model saved to: {MODEL_SAVE_PATH}")

## 11. Plot Training and Validation Curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_acc"], label="Train Accuracy")
plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

## 12. Load Best Model for Testing

In [ ]:
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()
print("Best model loaded successfully.")

## 13. Evaluate on Test Set

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

test_acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average='weighted'
)

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1-score      : {f1:.4f}")

## 14. Detailed Classification Report

In [ ]:
print(classification_report(all_labels, all_preds, target_names=class_names))

## 15. Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 16. Per-Class Performance Analysis

This helps identify:
- **Best performing class**
- **Worst performing class**

We compute per-class F1-score and accuracy-like recall behavior.

In [ ]:
report = classification_report(
    all_labels,
    all_preds,
    target_names=class_names,
    output_dict=True
)

class_f1 = {cls: report[cls]['f1-score'] for cls in class_names}
class_recall = {cls: report[cls]['recall'] for cls in class_names}
class_precision = {cls: report[cls]['precision'] for cls in class_names}

best_class = max(class_f1, key=class_f1.get)
worst_class = min(class_f1, key=class_f1.get)

print("Per-class F1-scores:")
for cls, score in class_f1.items():
    print(f"{cls:12s}: {score:.4f}")

print("\nBest performing class :", best_class, f"({class_f1[best_class]:.4f})")
print("Worst performing class:", worst_class, f"({class_f1[worst_class]:.4f})")

## 17. Bar Chart of Per-Class F1 Scores

In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(class_f1.keys(), class_f1.values())
plt.xticks(rotation=45)
plt.ylabel("F1-score")
plt.title("Per-Class F1-score")
plt.tight_layout()
plt.show()

## 18. Discussion and Analysis

### Observations
- The model learns meaningful image representations using stacked convolution layers.
- Batch normalization improves training stability.
- Dropout reduces overfitting.
- Data augmentation helps generalization on unseen test images.

### Best and Worst Classes
- The **best class** usually contains more visually distinct features.
- The **worst class** often overlaps with visually similar categories.

### Rationale for Hyperparameters
- **Adam** was chosen for faster and stable convergence.
- **StepLR scheduler** reduces the learning rate gradually for better refinement.
- **Batch size 128** balances speed and memory efficiency.
- **15 epochs** is enough for a good assignment-level result, though more epochs may improve performance.

### Possible Future Improvements
- Deeper CNN architecture
- Residual connections
- More advanced augmentation
- Hyperparameter tuning
- Transfer learning with pretrained models

## 19. Save Final Model Weights

In [ ]:
torch.save(model.state_dict(), "cnn_cifar10_final.pth")
print("Final model weights saved as cnn_cifar10_final.pth")

## 20. Conclusion

In this assignment, a custom CNN was developed to classify CIFAR-10 images. The notebook includes:
- data loading and preprocessing
- CNN architecture design
- training and validation
- performance visualization
- evaluation metrics
- confusion matrix
- per-class performance analysis
- saved model weights

This satisfies the complete assignment workflow.